<a href="https://colab.research.google.com/github/asiddiqha/rag-huggingface-docs/blob/main/RAG_Pipeline_Ayesha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation over Hugging Face Documentation

**Ayesha Siddiqha** · built during my GenAI coursework (Scaler)

I built an end-to-end RAG pipeline that answers questions about the Hugging Face documentation, cites its sources, and declines questions it cannot answer from the documents.

| Stage | My choice | Why |
|---|---|---|
| Knowledge base | `m-ric/huggingface_doc` (2,647 docs) | Real technical documentation with source paths for citations |
| Chunking | Chonkie, fixed 256 tokens + 32 overlap | Fast and predictable at 33k chunks; I compared 6 strategies first |
| Embeddings | `BAAI/bge-base-en-v1.5` | Strong retrieval quality at 768 dimensions; MiniLM used as a fast baseline |
| Vector DB | ChromaDB with HNSW, cosine | Runs in-process, has metadata filtering, no paid service needed |
| Query enhancement | HyDE (optional) | Closes the gap between short questions and long documents |
| Reranking | Cross-encoder `ms-marco-MiniLM-L-6-v2` | Precise top-k after a fast, recall-oriented first stage |
| Generation | GPT-OSS 120B on Groq, temperature 0 | Fast hosted inference; grounded prompt with citations |

**To run:** Colab with a T4 GPU, and a Groq API key stored in Colab Secrets as `groq`.

## 0. Setup

In [1]:
# All dependencies in one place
!pip install -q groq "chonkie[viz,groq]" rich sentence-transformers chromadb \
    langchain langchain-core langchain-chroma langchain-huggingface datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 991.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.4/386.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

In [3]:
import os
import time
from google.colab import userdata
from groq import Groq

# I read the API key from Colab Secrets so it never appears in the notebook.
# If the secret is missing, I fall back to a manual input prompt.
try:
    os.environ["GROQ_API_KEY"] = userdata.get("groq")
except Exception:
    import getpass
    print("Groq key not found in Colab Secrets (name: 'groq'). Enter it below.")
    raw_key = getpass.getpass("Groq API key: ")
    # Convert explicitly to string to avoid TypeError if getpass returns a dict-like structure
    os.environ["GROQ_API_KEY"] = str(raw_key)

groq_client = Groq()

# Groq retired Llama 3.3 70B and Llama 3.1 8B on 16 Aug 2026; I moved to their recommended replacements.
LLM_MODEL = "openai/gpt-oss-120b"
FALLBACK_LLM_MODEL = "openai/gpt-oss-20b"
print("Models available to my key:", sorted(m.id for m in groq_client.models.list().data))

def chat(messages, **kwargs):
    """Call the main model and fall back to the smaller one if it is unavailable.
    GPT-OSS is a reasoning model, so I keep reasoning effort low to leave tokens for the answer."""
    kwargs.setdefault("reasoning_effort", "low")
    try:
        return groq_client.chat.completions.create(model=LLM_MODEL, messages=messages, **kwargs)
    except Exception as e:
        print(f"{LLM_MODEL} failed ({type(e).__name__}); using {FALLBACK_LLM_MODEL}")
        return groq_client.chat.completions.create(model=FALLBACK_LLM_MODEL, messages=messages, **kwargs)

# Run settings
NUM_CHUNKS_TO_INDEX = None    # None = index every chunk (about 33k; a few minutes on a T4). Use e.g. 2000 on CPU.
RUN_SLOW_LLM_CHUNKER = False  # LLM-based chunking makes many API calls (about 10 minutes)

Models available to my key: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


## 1. Knowledge base

An LLM on its own has a knowledge cutoff, no access to private documents, a limited context window, and a tendency to hallucinate. RAG addresses all four by retrieving relevant passages and grounding the answer in them.

I use the Hugging Face documentation dataset: 2,647 documents, each with its source file path, which I keep for citations.

In [4]:
from datasets import load_dataset

dataset = load_dataset("m-ric/huggingface_doc", split="train")
print(f"Documents: {len(dataset)} | Fields: {dataset.features}")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

huggingface_doc.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Documents: 2647 | Fields: {'text': Value('string'), 'source': Value('string')}


In [5]:
example = dataset[0]
print("Source:", example["source"])
print("-" * 50)
print(example["text"][:1000])

Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
--------------------------------------------------
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-documentation/main/assets/1_repository.png" alt="select repository" />

## 2. Select your Cloud Provider and region. Initially, only AWS will be available as a Cloud Provider with the `us-east-1` and `eu-west-1` regions. We will add Azure soon, and if you need to test Endpoints with other Cloud Providers or regions, please let us know.

<img src="https://

## 2. Baseline: the LLM without retrieval
A quick check that the API works before I add retrieval.

In [6]:
response = chat([{"role": "user", "content": "What is the capital of France?"}])
print(response.choices[0].message.content)

The capital of France is **Paris**.


## 3. Chunking strategies

Chunk size is a trade-off: large chunks dilute the embedding and use up the context window; small chunks lose surrounding context. I compared six strategies on one sample article before choosing one for the full dataset.

In [7]:
# Sample article used to compare chunking strategies
teaching_document = """
The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions, the ability to focus deeply on cognitively demanding tasks has become increasingly rare—and increasingly valuable. This is what author Cal Newport calls "deep work": professional activities performed in a state of distraction-free concentration that push your cognitive capabilities to their limit.

Why Deep Work Matters

Deep work is valuable for several reasons. First, it allows you to produce high-quality output in less time. When you're fully concentrated on a single task, you work more efficiently and make fewer mistakes. Second, deep work helps you master complex skills faster. Learning difficult concepts requires sustained attention—something impossible to achieve when constantly switching between tasks.

Creating Your Deep Work Environment

The first step in cultivating deep work is designing an environment that supports concentration. This means more than just finding a quiet space. Consider your physical setup: Is your desk organized? Is the lighting appropriate? Do you have everything you need within reach?

Digital distractions are equally important to address. Turn off notifications on your devices. Close unnecessary browser tabs. Use website blockers if needed. The goal is to create a space where your attention isn't constantly being pulled away from the task at hand.

Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial. Don't wait for free time to magically appear—it won't. Instead, treat deep work sessions as important appointments with yourself. Many people find that early morning hours work best, when their mental energy is highest and distractions are minimal.

Start with manageable blocks. If you're new to deep work, even 60-90 minutes of focused time can feel challenging. As you build your concentration muscles, gradually extend these sessions. Some professionals work up to four-hour blocks of uninterrupted deep work.

The Shutdown Ritual

Just as important as starting deep work is knowing when to stop. Develop a shutdown ritual to mark the end of your workday. This might include reviewing your task list for tomorrow, closing all work-related browser tabs, and saying a specific phrase like "shutdown complete."

This ritual serves multiple purposes. It helps you mentally disconnect from work, reduces anxiety about unfinished tasks, and ensures you've captured anything important before stepping away. Without this clear boundary, work thoughts tend to linger into your evening, preventing true rest and recovery.

Measuring Your Progress

Track your deep work hours each week. This simple metric provides valuable feedback on your habits. You might discover that you're spending less time in deep work than you thought, or that certain days of the week are more conducive to concentration than others.

Remember, the goal isn't to spend every waking hour in deep work. Even the most focused professionals typically max out at four to five hours of truly deep work per day. What matters is consistency and intentionality—making deep work a regular part of your routine rather than an occasional occurrence.
"""

print(f"Length: {len(teaching_document)} characters, {teaching_document.count(chr(10) + chr(10))} paragraphs")

Length: 3219 characters, 15 paragraphs


In [8]:
from chonkie import TokenChunker, Visualizer
from tokenizers import Tokenizer

# GPT-2 tokenizer as a standard reference for counting tokens
tokenizer = Tokenizer.from_pretrained("gpt2")
viz = Visualizer()

def show_chunks(name, chunks):
    """Print chunk count and sizes, then colour-code the chunks in the text."""
    print(f"{name}: {len(chunks)} chunks, sizes (tokens) = {[c.token_count for c in chunks]}")
    viz.print(chunks)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

### 3.1 Fixed size
Simplest and fastest. Boundaries can fall mid-sentence.

In [9]:
fixed_chunker = TokenChunker(tokenizer=tokenizer, chunk_size=128, chunk_overlap=0)
show_chunks("Fixed size", fixed_chunker.chunk(teaching_document))

Fixed size: 5 chunks, sizes (tokens) = [128, 128, 128, 128, 105]


The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions, the ability to focus deeply on cognitively 
demanding tasks has become increasingly rare—and increasingly valuable. This is what author Cal Newport calls "deep
work": professional activities performed in a state of distraction-free concentration that push your cognitive 
capabilities to their limit.

Why Deep Work Matters

Deep work is valuable for several reasons. First, it allows you to produce high-quality output in less time. When 
you're fully concentrated on a single task, you work more efficiently and make fewer mistakes. Second, deep work 
helps you master complex skills faster. Learning difficult concepts requires sustained attention—something 
impossible to achieve when constantly switching between tasks.

Creating Your Deep Work Environment

The first step in cultivating deep work is designing an environment that supports concentration. This means more 
than just finding a quiet space. Consider your physical setup: Is your desk organized? Is the lighting appropriate?
Do you have everything you need within reach?

Digital distractions are equally important to address. Turn off notifications on your devices. Close unnecessary 
browser tabs. Use website blockers if needed. The goal is to create a space where your attention isn't constantly 
being pulled away from the task at hand.

Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial. Don't wait for free time to magically appear—it won't.
Instead, treat deep work sessions as important appointments with yourself. Many people find that early morning 
hours work best, when their mental energy is highest and distractions are minimal.

Start with manageable blocks. If you're new to deep work, even 60-90 minutes of focused time can feel challenging. 
As you build your concentration muscles, gradually extend these sessions. Some professionals work up to four-hour 
blocks of uninterrupted deep work.

The Shutdown Ritual

Just as important as starting deep work is knowing when to stop. Develop a shutdown ritual to mark the end of your 
workday. This might include reviewing your task list for tomorrow, closing all work-related browser tabs, and 
saying a specific phrase like "shutdown complete."

This ritual serves multiple purposes. It helps you mentally disconnect from work, reduces anxiety about unfinished 
tasks, and ensures you've captured anything important before stepping away. Without this clear boundary, work 
thoughts tend to linger into your evening, preventing true rest and recovery.

Measuring Your Progress

Track your deep work hours each week. This simple metric provides valuable feedback on your habits. You might 
discover that you're spending less time in deep work than you thought, or that certain days of the week are more 
conducive to concentration than others.

Remember, the goal isn't to spend every waking hour in deep work. Even the most focused professionals typically max
out at four to five hours of truly deep work per day. What matters is consistency and intentionality—making deep 
work a regular part of your routine rather than an occasional occurrence.

### 3.2 Fixed size with overlap
25% overlap keeps sentences that are cut at a boundary intact in one of the two chunks.

In [10]:
overlap_chunker = TokenChunker(tokenizer=tokenizer, chunk_size=128, chunk_overlap=32)
show_chunks("Fixed size + overlap", overlap_chunker.chunk(teaching_document))

Fixed size + overlap: 7 chunks, sizes (tokens) = [128, 128, 128, 128, 128, 128, 41]


The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions, the ability to focus deeply on cognitively 
demanding tasks has become increasingly rare—and increasingly valuable. This is what author Cal Newport calls "deep
work": professional activities performed in a state of distraction-free concentration that push your cognitive 
capabilities to their limit.

Why Deep Work Matters

Deep work is valuable for several reasons. First, it allows you to produce high-quality output in less time. When 
you're fully concentrated on a single task, you work more efficiently and make fewer mistakes. Second, deep work 
helps you master complex skills faster. Learning difficult concepts requires sustained attention—something 
impossible to achieve when constantly switching between tasks.

Creating Your Deep Work Environment

The first step in cultivating deep work is designing an environment that supports concentration. This means more 
than just finding a quiet space. Consider your physical setup: Is your desk organized? Is the lighting appropriate?
Do you have everything you need within reach?

Digital distractions are equally important to address. Turn off notifications on your devices. Close unnecessary 
browser tabs. Use website blockers if needed. The goal is to create a space where your attention isn't constantly 
being pulled away from the task at hand.

Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial. Don't wait for free time to magically appear—it won't.
Instead, treat deep work sessions as important appointments with yourself. Many people find that early morning 
hours work best, when their mental energy is highest and distractions are minimal.

Start with manageable blocks. If you're new to deep work, even 60-90 minutes of focused time can feel challenging. 
As you build your concentration muscles, gradually extend these sessions. Some professionals work up to four-hour 
blocks of uninterrupted deep work.

The Shutdown Ritual

Just as important as starting deep work is knowing when to stop. Develop a shutdown ritual to mark the end of your 
workday. This might include reviewing your task list for tomorrow, closing all work-related browser tabs, and 
saying a specific phrase like "shutdown complete."

This ritual serves multiple purposes. It helps you mentally disconnect from work, reduces anxiety about unfinished 
tasks, and ensures you've captured anything important before stepping away. Without this clear boundary, work 
thoughts tend to linger into your evening, preventing true rest and recovery.

Measuring Your Progress

Track your deep work hours each week. This simple metric provides valuable feedback on your habits. You might 
discover that you're spending less time in deep work than you thought, or that certain days of the week are more 
conducive to concentration than others.

Remember, the goal isn't to spend every waking hour in deep work. Even the most focused professionals typically max
out at four to five hours of truly deep work per day. What matters is consistency and intentionality—making deep 
work a regular part of your routine rather than an occasional occurrence.

### 3.3 Semantic
Splits where the topic shifts, based on sentence-embedding similarity. Better boundaries for prose, at the cost of an embedding pass.

In [11]:
from chonkie import SemanticChunker

semantic_chunker = SemanticChunker(
    embedding_model="minishlab/potion-base-32M",  # small static-embedding model, fast enough for chunking
    threshold=0.7,                                # similarity below this starts a new chunk
    chunk_size=512,
    similarity_window=3,                          # sentences compared at a time
)
show_chunks("Semantic", semantic_chunker.chunk(teaching_document))

2026-10-19 15:43:01,191 | WARNING  | chonkie.embeddings.auto:get_embeddings:98 - Failed to load minishlab/potion-base-32M with Model2VecEmbeddings: model2vec is not available. Please install it via `pip install chonkie[model2vec]`
Falling back to loading default provider model.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/chonkie/embeddings/model2vec.py", line 32, in __init__
    from model2vec import StaticModel
ModuleNotFoundError: No module named 'model2vec'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/chonkie/embeddings/auto.py", line 96, in get_embeddings
    embeddings_instance = cast(Any, embeddings_cls)(model, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/chonkie/embeddings/model2vec.py", line 34, in __init__
    raise ImportError(
        "model2vec is not available. Please install it via `pip install chonkie[model2vec]`

modules.json:   0%|          | 0.00/278 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

./model.safetensors: reconstructing file:   0%|          |  0.00B /  129MB            

./model.safetensors: downloading bytes:           |  0.00B            

Semantic: 12 chunks, sizes (tokens) = [84, 93, 15, 9, 8, 18, 88, 109, 52, 3, 8, 101]


/usr/local/lib/python3.13/dist-packages/chonkie/embeddings/sentence_transformer.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()  # ty: ignore[deprecated]


The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions, the ability to focus deeply on cognitively 
demanding tasks has become increasingly rare—and increasingly valuable. This is what author Cal Newport calls "deep
work": professional activities performed in a state of distraction-free concentration that push your cognitive 
capabilities to their limit.

Why Deep Work Matters

Deep work is valuable for several reasons. First, it allows you to produce high-quality output in less time. When 
you're fully concentrated on a single task, you work more efficiently and make fewer mistakes. Second, deep work 
helps you master complex skills faster. Learning difficult concepts requires sustained attention—something 
impossible to achieve when constantly switching between tasks.

Creating Your Deep Work Environment

The first step in cultivating deep work is designing an environment that supports concentration. This means more 
than just finding a quiet space. Consider your physical setup: Is your desk organized? Is the lighting appropriate?
Do you have everything you need within reach?

Digital distractions are equally important to address. Turn off notifications on your devices. Close unnecessary 
browser tabs. Use website blockers if needed. The goal is to create a space where your attention isn't constantly 
being pulled away from the task at hand.

Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial. Don't wait for free time to magically appear—it won't.
Instead, treat deep work sessions as important appointments with yourself. Many people find that early morning 
hours work best, when their mental energy is highest and distractions are minimal.

Start with manageable blocks. If you're new to deep work, even 60-90 minutes of focused time can feel challenging. 
As you build your concentration muscles, gradually extend these sessions. Some professionals work up to four-hour 
blocks of uninterrupted deep work.

The Shutdown Ritual

Just as important as starting deep work is knowing when to stop. Develop a shutdown ritual to mark the end of your 
workday. This might include reviewing your task list for tomorrow, closing all work-related browser tabs, and 
saying a specific phrase like "shutdown complete."

This ritual serves multiple purposes. It helps you mentally disconnect from work, reduces anxiety about unfinished 
tasks, and ensures you've captured anything important before stepping away. Without this clear boundary, work 
thoughts tend to linger into your evening, preventing true rest and recovery.

Measuring Your Progress

Track your deep work hours each week. This simple metric provides valuable feedback on your habits. You might 
discover that you're spending less time in deep work than you thought, or that certain days of the week are more 
conducive to concentration than others.

Remember, the goal isn't to spend every waking hour in deep work. Even the most focused professionals typically max
out at four to five hours of truly deep work per day. What matters is consistency and intentionality—making deep 
work a regular part of your routine rather than an occasional occurrence.

### 3.4 Recursive
Follows the document hierarchy (headings, paragraphs, sentences). Suited to structured Markdown like documentation.

In [12]:
from chonkie import RecursiveChunker

markdown_document = """
# The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions...

## Why Deep Work Matters

Deep work is valuable for several reasons...

## Creating Your Deep Work Environment

The first step in cultivating deep work...

### Physical Setup

Consider your physical setup: Is your desk organized?...

### Digital Distractions

Digital distractions are equally important to address...

## Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial...
"""

# A deliberately small chunk size so the hierarchy-based splits are visible
recursive_chunker = RecursiveChunker(tokenizer=tokenizer, chunk_size=15)
show_chunks("Recursive", recursive_chunker.chunk(markdown_document))

Recursive: 9 chunks, sizes (tokens) = [14, 12, 15, 7, 14, 12, 14, 8, 14]


# The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions...

## Why Deep Work Matters

Deep work is valuable for several reasons...

## Creating Your Deep Work Environment

The first step in cultivating deep work...

### Physical Setup

Consider your physical setup: Is your desk organized?...

### Digital Distractions

Digital distractions are equally important to address...

## Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial...

### 3.5 Late chunking
Embeds the whole document first, then splits, so each chunk's embedding keeps document-level context. Useful for long documents such as contracts.

In [13]:
from chonkie import LateChunker

late_chunker = LateChunker(embedding_model="minishlab/potion-base-32M", chunk_size=512)
show_chunks("Late", late_chunker.chunk(teaching_document))

Late: 2 chunks, sizes (tokens) = [1, 1]


/usr/local/lib/python3.13/dist-packages/chonkie/embeddings/sentence_transformer.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()  # ty: ignore[deprecated]


The Art of Deep Work: A Guide to Productivity

In today's world of constant notifications and endless distractions, the ability to focus deeply on cognitively 
demanding tasks has become increasingly rare—and increasingly valuable. This is what author Cal Newport calls "deep
work": professional activities performed in a state of distraction-free concentration that push your cognitive 
capabilities to their limit.

Why Deep Work Matters

Deep work is valuable for several reasons. First, it allows you to produce high-quality output in less time. When 
you're fully concentrated on a single task, you work more efficiently and make fewer mistakes. Second, deep work 
helps you master complex skills faster. Learning difficult concepts requires sustained attention—something 
impossible to achieve when constantly switching between tasks.

Creating Your Deep Work Environment

The first step in cultivating deep work is designing an environment that supports concentration. This means more 
than just finding a quiet space. Consider your physical setup: Is your desk organized? Is the lighting appropriate?
Do you have everything you need within reach?

Digital distractions are equally important to address. Turn off notifications on your devices. Close unnecessary 
browser tabs. Use website blockers if needed. The goal is to create a space where your attention isn't constantly 
being pulled away from the task at hand.

Time Blocking for Deep Work

Scheduling specific blocks of time for deep work is crucial. Don't wait for free time to magically appear—it won't.
Instead, treat deep work sessions as important appointments with yourself. Many people find that early morning 
hours work best, when their mental energy is highest and distractions are minimal.

Start with manageable blocks. If you're new to deep work, even 60-90 minutes of focused time can feel challenging. 
As you build your concentration muscles, gradually extend these sessions. Some professionals work up to four-hour 
blocks of uninterrupted deep work.

The Shutdown Ritual

Just as important as starting deep work is knowing when to stop. Develop a shutdown ritual to mark the end of your 
workday. This might include reviewing your task list for tomorrow, closing all work-related browser tabs, and 
saying a specific phrase like "shutdown complete."

This ritual serves multiple purposes. It helps you mentally disconnect from work, reduces anxiety about unfinished 
tasks, and ensures you've captured anything important before stepping away. Without this clear boundary, work 
thoughts tend to linger into your evening, preventing true rest and recovery.

Measuring Your Progress

Track your deep work hours each week. This simple metric provides valuable feedback on your habits. You might 
discover that you're spending less time in deep work than you thought, or that certain days of the week are more 
conducive to concentration than others.

Remember, the goal isn't to spend every waking hour in deep work. Even the most focused professionals typically max
out at four to five hours of truly deep work per day. What matters is consistency and intentionality—making deep 
work a regular part of your routine rather than an occasional occurrence.

### 3.6 LLM-based
An LLM decides the boundaries. Best quality, but slow and costly, so it is off by default.

In [14]:
from chonkie import SlumberChunker
from chonkie.genie import GroqGenie

if RUN_SLOW_LLM_CHUNKER:
    llm_chunker = SlumberChunker(
        genie=GroqGenie(model=LLM_MODEL),
        tokenizer="gpt2",
        chunk_size=512,
        candidate_size=128,           # text window the LLM sees per split decision
        min_characters_per_chunk=50,
    )
    show_chunks("LLM-based", llm_chunker.chunk(teaching_document))
else:
    print("Skipped: set RUN_SLOW_LLM_CHUNKER = True to run.")

Skipped: set RUN_SLOW_LLM_CHUNKER = True to run.


## 4. Chunking the knowledge base and embedding

For the full dataset I use fixed-size chunks of 256 tokens with 32 overlap. Documentation sections are short and well structured, and fixed-size chunking is fast and predictable at this scale. Semantic or LLM chunking would add compute for a small gain here.

I keep each chunk's source path as metadata so answers can cite where they came from.

In [15]:
kb_chunker = TokenChunker(tokenizer=tokenizer, chunk_size=256, chunk_overlap=32)

chunk_texts, chunk_sources = [], []
for doc in dataset:
    for chunk in kb_chunker.chunk(doc["text"]):
        chunk_texts.append(chunk.text)
        chunk_sources.append(doc["source"])

n = len(chunk_texts) if NUM_CHUNKS_TO_INDEX is None else min(NUM_CHUNKS_TO_INDEX, len(chunk_texts))
print(f"Chunks: {len(chunk_texts)} from {len(dataset)} documents | indexing: {n}")

Chunks: 33769 from 2647 documents | indexing: 33769


In [16]:
from sentence_transformers import SentenceTransformer

# Baseline embedding model: all-MiniLM-L6-v2 (384 dimensions), small and fast
minilm = SentenceTransformer("all-MiniLM-L6-v2")
dim = minilm.get_embedding_dimension() if hasattr(minilm, "get_embedding_dimension") else minilm.get_sentence_embedding_dimension()
print(f"Embedding dimension: {dim}")

chunk_embeddings = minilm.encode(chunk_texts[:n], batch_size=128, show_progress_bar=True)
print(f"Embeddings: {chunk_embeddings.shape}")

# Queries must be embedded with the same model as the chunks
query_embedding = minilm.encode("How do I load a pretrained model?")
print(f"Query embedding: {query_embedding.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


Batches:   0%|          | 0/264 [00:00<?, ?it/s]

Embeddings: (33769, 384)
Query embedding: (384,)


## 5. Vector database with ChromaDB

I chose ChromaDB because it runs inside the notebook, stores text, vectors and metadata together, and indexes with HNSW out of the box. FAISS is a search library without a storage or metadata layer, and managed options like Pinecone add cost and setup that this project does not need.

In [17]:
import chromadb

chroma_client = chromadb.Client()   # in-memory for this notebook; PersistentClient(path=...) keeps it on disk
if "huggingface_docs" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("huggingface_docs")

# Cosine distance instead of Chroma's default L2, to match how sentence embeddings are compared
collection = chroma_client.create_collection(name="huggingface_docs", metadata={"hnsw:space": "cosine"})

# Chroma limits the size of a single insert, so I add in batches
BATCH = 5000
for start in range(0, n, BATCH):
    end = min(start + BATCH, n)
    collection.add(
        ids=[f"doc_{i}" for i in range(start, end)],
        documents=chunk_texts[start:end],
        embeddings=chunk_embeddings[start:end].tolist(),
        metadatas=[{"chunk_index": i, "source": chunk_sources[i]} for i in range(start, end)],
    )
print(f"Collection size: {collection.count()}")

Collection size: 33769


In [18]:
def show_chroma_results(results, chars=300):
    """Print distance, chunk index, source and a text preview for each hit."""
    for i, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), 1):
        print(f"Result {i} | distance {dist:.4f} | chunk {meta['chunk_index']} | {meta['source']}")
        print(doc[:chars] + "...")
        print("-" * 80)

user_query = "How do I load a pretrained model?"
results = collection.query(query_embeddings=[minilm.encode(user_query).tolist()], n_results=3)
print(f"Query: {user_query}\n")
show_chroma_results(results)

Query: How do I load a pretrained model?

Result 1 | distance 0.3603 | chunk 33155 | huggingface/course/blob/main/subtitles/en/raw/chapter4/03b_push-to-hub-pt.md
 the from_pretrained method. We just have to use the identifier from the hub and we can see that the model configuration and weights are automatically downloaded. We can use this model as we would any other Transformers model, for instance by loading it in a pipeline. Try the push_to_hub API on your...
--------------------------------------------------------------------------------
Result 2 | distance 0.3731 | chunk 9906 | huggingface/blog/blob/main/how-to-train.md
models and everyone can load it using `AutoModel.from_pretrained("username/model_name")`.

[![tb](assets/01_how-to-train/model_page.png)](https://huggingface.co/julien-c/EsperBERTo-small)


If you want to take a look at models in different languages, check https://huggingface.co/models

[![all model...
----------------------------------------------------------------

### Metadata filtering
I restrict the search space with a metadata condition before similarity search. In production the same mechanism filters by product, version or date.

In [19]:
filtered = collection.query(
    query_embeddings=[minilm.encode("loading models").tolist()],
    n_results=3,
    where={"source": {"$in": sorted({s for s in chunk_sources[:n] if "transformers" in s})}},  # Transformers docs only
)
show_chroma_results(filtered, chars=200)

Result 1 | distance 0.4800 | chunk 22405 | huggingface/transformers/blob/main/docs/source/en/add_new_model.md
 our experience, however, we
found that the fundamental design choices and philosophies of the library are crucial to efficiently scale 🤗
Transformers while keeping maintenance costs at a reasonable l...
--------------------------------------------------------------------------------
Result 2 | distance 0.4812 | chunk 30142 | huggingface/transformers/blob/main/docs/source/en/philosophy.md
-on practitioners who want to fine-tune those models or serve them in production, or both.
- engineers who just want to download a pretrained model and use it to solve a given machine learning task.

...
--------------------------------------------------------------------------------
Result 3 | distance 0.4831 | chunk 6189 | huggingface/transformers/blob/main/docs/source/en/main_classes/model.md
 methods for loading/saving a model either from a local
file or directory, or from a pretrained mod

## 6. Production retriever: bge embeddings with HNSW

For the main pipeline I switch to `BAAI/bge-base-en-v1.5`, which retrieves noticeably better than MiniLM on retrieval benchmarks, at the cost of a larger model (768 dimensions). I normalise the embeddings so cosine similarity and dot product rank documents the same way.

The retriever is a bi-encoder: queries and documents are embedded separately, so document vectors are computed once and search stays fast. HNSW gives approximate nearest-neighbour search in roughly logarithmic time, instead of comparing the query with every vector.

In [20]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

bge = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True, "batch_size": 128},
)

docs = [Document(page_content=chunk_texts[i], metadata={"chunk_id": i, "source": chunk_sources[i]}) for i in range(n)]

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=bge,
    collection_name="huggingface_docs_bge",
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Indexed {vectorstore._collection.count()} chunks (HNSW, cosine)")

# Stage 1 favours recall: I retrieve 20 candidates and let the reranker pick the best few
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 20})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 33769 chunks (HNSW, cosine)


In [21]:
test_query = "How do I load a pretrained model?"

start = time.time()
results = retriever.invoke(test_query)
print(f"Search time: {(time.time() - start) * 1000:.1f} ms | candidates: {len(results)}\n")

for i, doc in enumerate(results[:3], 1):
    print(f"Result {i} | chunk {doc.metadata['chunk_id']} | {doc.metadata['source']}")
    print(doc.page_content[:200] + "...")
    print("-" * 80)

Search time: 61.3 ms | candidates: 20

Result 1 | chunk 18253 | huggingface/pytorch-image-models/blob/main/hfdocs/source/models/regnetx.mdx
 load a pretrained model:

```py
>>> import timm
>>> model = timm.create_model('regnetx_002', pretrained=True)
>>> model.eval()
```

To load and preprocess the image:

```py
>>> import urllib
>>> from...
--------------------------------------------------------------------------------
Result 2 | chunk 9906 | huggingface/blog/blob/main/how-to-train.md
models and everyone can load it using `AutoModel.from_pretrained("username/model_name")`.

[![tb](assets/01_how-to-train/model_page.png)](https://huggingface.co/julien-c/EsperBERTo-small)


If you wan...
--------------------------------------------------------------------------------
Result 3 | chunk 3570 | huggingface/transformers/blob/main/docs/source/en/quicktour.md
<tf>
🤗 Transformers provides a simple and unified way to load pretrained instances. This means you can load an [`TFAutoModel`] like you 

## 7. HyDE (Hypothetical Document Embeddings)

A short question rarely looks like the long passage that answers it. With HyDE, I ask the LLM to draft a hypothetical answer in documentation style and search with that draft instead. I keep it optional and compare it with standard search, because an inaccurate draft can pull retrieval off topic.

In [22]:
def generate_hypothetical_document(query):
    """Draft a documentation-style answer to use as the search text.
    Temperature 0.7 gives a natural, varied draft; it is only used for retrieval, never shown as the answer."""
    response = chat(
        [
            {"role": "system", "content": "You are a technical documentation writer for Hugging Face. "
                                          "Write a concise, technical answer in the style of official documentation."},
            {"role": "user", "content": f"Answer this question: {query}"},
        ],
        max_tokens=800,
        temperature=0.7,
    )
    return response.choices[0].message.content


def hyde_search(query, retriever):
    """Retrieve with the hypothetical answer instead of the raw query."""
    hypothetical_doc = generate_hypothetical_document(query)
    return retriever.invoke(hypothetical_doc), hypothetical_doc


test_query = "How do I load a pretrained model?"

print("STANDARD SEARCH")
for i, doc in enumerate(retriever.invoke(test_query)[:3], 1):
    print(f"{i}. [{doc.metadata['source']}] {doc.page_content[:150]}...")

hyde_results, hypothetical_doc = hyde_search(test_query, retriever)
print("\nHYPOTHETICAL DOCUMENT\n" + hypothetical_doc[:600])
print("\nHyDE SEARCH")
for i, doc in enumerate(hyde_results[:3], 1):
    print(f"{i}. [{doc.metadata['source']}] {doc.page_content[:150]}...")

STANDARD SEARCH
1. [huggingface/pytorch-image-models/blob/main/hfdocs/source/models/regnetx.mdx]  load a pretrained model:

```py
>>> import timm
>>> model = timm.create_model('regnetx_002', pretrained=True)
>>> model.eval()
```

To load and prepr...
2. [huggingface/blog/blob/main/how-to-train.md] models and everyone can load it using `AutoModel.from_pretrained("username/model_name")`.

[![tb](assets/01_how-to-train/model_page.png)](https://hugg...
3. [huggingface/transformers/blob/main/docs/source/en/quicktour.md] <tf>
🤗 Transformers provides a simple and unified way to load pretrained instances. This means you can load an [`TFAutoModel`] like you would load an ...

HYPOTHETICAL DOCUMENT
**Loading a pretrained model**

Hugging Face 🤗 Transformers provides a single, unified API to download and instantiate any model that has been uploaded to the 🤗 Model Hub. The typical workflow is:

```python
from transformers import AutoModel, AutoTokenizer   # or AutoModelForXXX, AutoModelForSeq2SeqL

## 8. Two-stage retrieval: bi-encoder, then cross-encoder

| | Bi-encoder (stage 1) | Cross-encoder (stage 2) |
|---|---|---|
| How it scores | query and document embedded separately | query and document read together |
| Speed | fast, vectors precomputed | slower, one model pass per pair |
| Role | narrow the whole index to 20 candidates | reorder those 20 and keep the top few |

I use `cross-encoder/ms-marco-MiniLM-L-6-v2`, trained on MS MARCO passage ranking and small enough to rerank 20 pairs in about 100 ms. A larger reranker such as bge-reranker would be more accurate but slower. ColBERT (late interaction) is another option I considered; it needs its own index, which is more than this project requires.

In [23]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_documents(query, documents, reranker=reranker, k=5):
    """Score each (query, chunk) pair and keep the top k.
    Scores are raw logits, so they can be negative; only the ordering matters."""
    scores = reranker.predict([[query, doc.page_content] for doc in documents])
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)[:k]
    return [{"content": doc.page_content, "source": doc.metadata.get("source", ""), "score": float(score)}
            for doc, score in ranked]


def two_stage_retrieval(query, retriever=retriever, reranker=reranker, k_final=5):
    """Stage 1: fast HNSW search for 20 candidates. Stage 2: cross-encoder rerank to k_final."""
    t0 = time.time()
    candidates = retriever.invoke(query)
    t1 = time.time()
    final = rerank_documents(query, candidates, reranker, k_final)
    t2 = time.time()
    print(f"Stage 1: {len(candidates)} candidates in {(t1 - t0) * 1000:.1f} ms")
    print(f"Stage 2: top {len(final)} in {(t2 - t1) * 1000:.1f} ms")
    return final, candidates


query = "How do I fine-tune a transformer model?"
reranked, initial = two_stage_retrieval(query)

print("\nSTAGE 1 (bi-encoder)")
for i, doc in enumerate(initial[:5], 1):
    print(f"{i}. [{doc.metadata['source']}] {doc.page_content[:150]}...")

print("\nSTAGE 2 (cross-encoder)")
for i, r in enumerate(reranked, 1):
    print(f"{i}. score {r['score']:.3f} | [{r['source']}] {r['content'][:150]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Stage 1: 20 candidates in 16.4 ms
Stage 2: top 5 in 96.5 ms

STAGE 1 (bi-encoder)
1. [huggingface/optimum/blob/main/docs/source/bettertransformer/tutorials/convert.mdx]  into any issue, do not hesitate to open an isse on GitHub!

## Training compatibility

You can now benefit from the `BetterTransformer` API for your ...
2. [huggingface/transformers/blob/main/README.md] ishra, Igor Molybog, Yixin Nie, Andrew Poulton, Jeremy Reizenstein, Rashi Rungta, Kalyan Saladi, Alan Schelten, Ruan Silva, Eric Michael Smith, Ranjan...
3. [huggingface/transformers/blob/main/README_ko.md] �하고, 원하는 데이터로 fine-tuning해 커뮤니티나 우리의 [모델 허브](https://huggingface.co/models)에 공유할 수 있도록 API를 제공합니다. 또한, 모델 구조를 정의하는 각 파이썬 모듈은 완전히 독립적이여서 연구 실험을 위해 손쉽게 ...
4. [huggingface/blog/blob/main/fine-tune-whisper.md]  to the [Google Colab](https://colab.research.google.com/github/sanchit-gandhi/notebooks/blob/main/fine_tune_whisper.ipynb) 
should you wish to try fi...
5. [huggingface/transformers/blob/main/docs/source/en/perf_

## 9. Retrieval pipeline: optional HyDE, bi-encoder, cross-encoder

In [24]:
def production_retrieval_pipeline(query, retriever=retriever, reranker=reranker,
                                  use_hyde=True, k_final=5, verbose=True):
    """HyDE only changes the search text. Reranking always uses the original question,
    so the final ranking reflects what the user actually asked."""
    search_text = generate_hypothetical_document(query) if use_hyde else query
    if verbose and use_hyde:
        print(f"HyDE draft: {search_text[:150]}...\n")

    t0 = time.time()
    candidates = retriever.invoke(search_text)
    final = rerank_documents(query, candidates, reranker, k_final)
    if verbose:
        print(f"{len(candidates)} candidates → top {len(final)} in {(time.time() - t0) * 1000:.1f} ms")
    return final


results_with_hyde = production_retrieval_pipeline("What are the best practices for model quantization?")
for i, r in enumerate(results_with_hyde, 1):
    print(f"\n{i}. score {r['score']:.3f} | {r['source']}")
    print(r["content"][:250] + "...")

HyDE draft: **Model Quantization – Best‑Practice Guide**  
*Version 1.2 – Hugging Face Documentation*  

---

### 1. Choose the Right Quantization Scheme
| Scheme...

20 candidates → top 5 in 183.0 ms

1. score 3.545 | huggingface/blog/blob/main/generative-ai-models-on-intel-cpu.md
bd>
  <img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/143_q8chat/pic1.png">
</kbd>
<br>*Source: SmoothQuant*

The best quantization techniques to date quantize activations token-wise, causing either trunca...

2. score 0.217 | huggingface/transformers/blob/main/docs/source/en/model_doc/qdqbert.md
ical
Evaluation](https://arxiv.org/abs/2004.09602) by Hao Wu, Patrick Judd, Xiaojie Zhang, Mikhail Isaev and Paulius
Micikevicius.

The abstract from the paper is the following:

*Quantization techniques can reduce the size of Deep Neural Networks an...

3. score -0.617 | huggingface/transformers/blob/main/docs/source/en/llm_tutorial_optimization.md
xiv.org/abs/2208.07339

## 10. Grounded generation

The prompt restricts the model to the retrieved documents, asks for citations, and tells it to say so when the answer is not there. I use temperature 0 so answers are deterministic and stay close to the sources.

In [25]:
def generate_answer(query, retrieved):
    """Build a grounded prompt from the reranked chunks and generate a cited answer."""
    context = "\n\n---\n\n".join(
        f"[Document {i}] (source: {r['source']})\n{r['content']}"
        for i, r in enumerate(retrieved, 1)
    )
    prompt = f"""Answer the question using ONLY the documents below.

Rules:
1. Every fact must come from the documents.
2. Cite the documents you use, e.g. [Document 2].
3. If the documents do not contain the answer, reply exactly: "The documents don't contain this information."
4. Do not add knowledge from outside the documents.

DOCUMENTS:
{context}

QUESTION: {query}

ANSWER:"""
    response = chat([{"role": "user", "content": prompt}], temperature=0.0, max_tokens=1500)
    return response.choices[0].message.content


def complete_rag_pipeline(query, use_hyde=True, k_final=5):
    """Retrieve, rerank and answer, then print the answer with its sources."""
    retrieved = production_retrieval_pipeline(query, use_hyde=use_hyde, k_final=k_final, verbose=False)
    answer = generate_answer(query, retrieved)

    print(f"QUESTION: {query}\n")
    print("ANSWER\n" + "=" * 80)
    print(answer)
    print("\nSOURCES\n" + "=" * 80)
    for i, r in enumerate(retrieved, 1):
        print(f"[Document {i}] {r['source']} (rerank score {r['score']:.3f})")
    return answer, retrieved

### Test 1: answerable question

In [26]:
answer, sources = complete_rag_pipeline("How do I load a pretrained model in Transformers?", use_hyde=False, k_final=3)

QUESTION: How do I load a pretrained model in Transformers?

ANSWER
You can load a pretrained model with the `AutoModel` family (or the TensorFlow equivalents) by calling the class’s `from_pretrained` method and passing the model identifier:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "your-model-name"
model = AutoModelForCausalLM.from_pretrained(model_id)
```

For TensorFlow you use the `TFAutoModel…` classes in the same way, e.g.:

```python
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")
```

These calls automatically download the checkpoint and instantiate the correct architecture for the specified task. [Document 1] [Document 3]

SOURCES
[Document 1] huggingface/transformers/blob/main/docs/source/en/quicktour.md (rerank score 8.747)
[Document 2] huggingface/transformers/blob/main/docs/source/en/quantization.md (rerank score 8.544)
[Document

### Test 2: the same question with HyDE

In [27]:
answer, sources = complete_rag_pipeline("How do I load a pretrained model in Transformers?", use_hyde=True, k_final=3)

QUESTION: How do I load a pretrained model in Transformers?

ANSWER
You can load a pretrained model with the `AutoModel` (or task‑specific `AutoModel…` class) by calling its `from_pretrained` method and passing the model identifier:

```python
from transformers import AutoModelForSequenceClassification   # or AutoModel, TFAutoModel, etc.

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
```

The same pattern works for TensorFlow models using `TFAutoModel…` as shown in the TensorFlow example【Document 1】, and for generic PyTorch models as shown in the PyTorch quick‑tour example【Document 3】.

SOURCES
[Document 1] huggingface/transformers/blob/main/docs/source/en/quicktour.md (rerank score 8.747)
[Document 2] huggingface/transformers/blob/main/docs/source/en/quantization.md (rerank score 8.544)
[Document 3] huggingface/transformers/blob/main/docs/source/en/quicktour.md (rerank score 8.189)


### Test 3: out-of-scope question
The system should decline rather than hallucinate.

In [28]:
answer, sources = complete_rag_pipeline("What is the weather in San Francisco?", use_hyde=False, k_final=3)

QUESTION: What is the weather in San Francisco?

ANSWER
The documents don't contain this information.

SOURCES
[Document 1] huggingface/blog/blob/main/informer.md (rerank score -6.652)
[Document 2] huggingface/blog/blob/main/autoformer.md (rerank score -7.267)
[Document 3] huggingface/course/blob/main/chapters/en/chapter7/7.mdx (rerank score -9.255)


## Summary and next steps

**What I built:** offline, documents are chunked, embedded with a bi-encoder and indexed in ChromaDB with HNSW and metadata. Online, the query is optionally expanded with HyDE, 20 candidates are retrieved, a cross-encoder reranks them to the top 3 to 5, and the LLM answers from those passages with citations, or declines.

**What I would add for production:**
- Hybrid search: BM25 alongside dense retrieval, merged with Reciprocal Rank Fusion, to catch exact terms such as API names.
- Evaluation: recall@k and MRR for retrieval; faithfulness and answer relevance for generation; latency and cost per query.
- Persistence and incremental indexing, plus caching of frequent queries and embeddings.